In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [2]:
from datasets import load_dataset
import transformers
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding
from transformers import Trainer, TrainingArguments

import re

In [3]:
ds = load_dataset("tyqiangz/multilingual-sentiments", "all")
ds

DatasetDict({
    train: Dataset({
        features: ['text', 'source', 'language', 'label'],
        num_rows: 270399
    })
    validation: Dataset({
        features: ['text', 'source', 'language', 'label'],
        num_rows: 10857
    })
    test: Dataset({
        features: ['text', 'source', 'language', 'label'],
        num_rows: 14465
    })
})

In [4]:
# Sampling a few text's and their labels

for i in range(10):
    print(f"Sample {i*10}: Text: {ds['train'][i*10]['text']}, Label: {ds['train'][i*10]['label']}")

Sample 0: Text: yang memerlukan pemerhatian dan tindakan serius, Label: 0
Sample 10: Text: @mkini_bm Tutup kilang pelabur akan ugut untuk lari ke negara lain..mana berani kerajaan selangor dgn taikun, Label: 2
Sample 20: Text: Sebab itu kita perlu kerja lebih gigih bukan saja untuk menang kerusi DAP, tapi pada masa sama turut bantu rakan-rakan Harapan., Label: 0
Sample 30: Text: Peluang hanya datang sekali sahaja seumur hidup bagi menyeimbangkan duniawi dan ukhrawi, bagi menjadi insan yang kamil dan aset kepada negara dan bukannya liabiliti, Label: 0
Sample 40: Text: Usahlah percaya wayang mereka ini (pembangkang) yang ingin menjadi hero, kononnya boleh bangunkan Kampung Baru dalam sekelip mata, Label: 2
Sample 50: Text: @YERIMxOL Capek jempol gak Nav??, Label: 1
Sample 60: Text: haih apa dia la teringat cerita lama tetiba, Label: 2
Sample 70: Text: Dalam hal ini, SPR telah mendapat jaminan daripada pihak Pos Malaysia berkaitan aspek keselamatan dan kerahsiaan dengan semua pengendalian

In [5]:
# Filtering samples - removing any links present

def filter_samples(example):
    example['text'] = re.sub(r'https://\S+','', example['text'])

    return example

In [6]:
ds_filtered = ds.map(filter_samples)

In [7]:
# Tokenizer
model_id = "meta-llama/Llama-3.2-1B"

tokenizer = AutoTokenizer.from_pretrained(model_id,  model_max_length=512)
tokenizer.pad_token = tokenizer.eos_token

In [8]:
def tokenize(example, tokenizer):
    example = tokenizer(example['text'], padding=False, truncation=True)

    return example

In [9]:
ds['train'][0]

{'text': 'yang memerlukan pemerhatian dan tindakan serius',
 'source': 'malaya',
 'language': 'malay',
 'label': 0}

In [10]:
tokenized_ds = ds_filtered.map(tokenize, remove_columns=['text', 'source'], 
                               batched=True,
                               num_proc=25,
                               fn_kwargs={"tokenizer": tokenizer})

In [11]:
tokenized_ds['train'][0]

{'language': 'malay',
 'label': 0,
 'input_ids': [128000,
  41345,
  1871,
  261,
  75,
  28824,
  281,
  41996,
  9379,
  1122,
  9279,
  259,
  485,
  19818,
  1446,
  9334],
 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [12]:
data_collator = DataCollatorWithPadding(tokenizer, padding=True)

In [13]:
model = AutoModelForSequenceClassification.from_pretrained(model_id,num_labels=3,
                                                           pad_token_id=tokenizer.eos_token_id)

model

Some weights of LlamaForSequenceClassification were not initialized from the model checkpoint at meta-llama/Llama-3.2-1B and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


LlamaForSequenceClassification(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 2048, padding_idx=128001)
    (layers): ModuleList(
      (0-15): 16 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=512, bias=False)
          (v_proj): Linear(in_features=2048, out_features=512, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (up_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (down_proj): Linear(in_features=8192, out_features=2048, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((20

In [14]:
model.config.id2label = {0:"Positive", 1:"Neutral", 2:"Negative"}

In [15]:
from peft import LoraConfig, TaskType

lora_config = LoraConfig(
    task_type= TaskType.SEQ_CLS,
    lora_alpha=32,
    lora_dropout= 0.05,
    r= 16, # lora attention rank
    target_modules=["q_proj", "v_proj"],
    inference_mode= False
)

In [16]:
from peft import get_peft_model

lora_model = get_peft_model(model, lora_config)
lora_model.print_trainable_parameters()


trainable params: 1,710,080 || all params: 1,237,530,624 || trainable%: 0.1382


In [17]:
import numpy as np
import evaluate

metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

In [18]:
# Performance of the model before finetuning
training_args = TrainingArguments(
    per_device_eval_batch_size=32,
    report_to= "none",
    fp16=True
)

trainer = Trainer(
    model= lora_model,
    args = training_args,
    eval_dataset= tokenized_ds['test'],
    compute_metrics= compute_metrics,
    data_collator= data_collator
)

No label_names provided for model class `PeftModelForSequenceClassification`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


In [19]:
metrics = trainer.evaluate()
print(metrics)


{'eval_loss': 1.6950145959854126, 'eval_model_preparation_time': 0.0044, 'eval_accuracy': 0.3231939163498099, 'eval_runtime': 30.5001, 'eval_samples_per_second': 474.261, 'eval_steps_per_second': 14.852}


In [20]:
training_args = TrainingArguments( output_dir='llama32_multilingual_ft_lora',
                                  eval_strategy="steps",
                                  eval_steps=1000,
                                  num_train_epochs=1,
                                  per_device_train_batch_size=16,
                                  per_device_eval_batch_size=32,
                                  bf16=False,
                                  fp16=True,
                                  tf32=False,
                                  gradient_accumulation_steps=1,
                                  adam_beta1=0.9,
                                  adam_beta2=0.999,
                                  learning_rate=2e-5,
                                  weight_decay=0.01,
                                  logging_dir='logs',
                                  logging_strategy="steps",
                                  logging_steps = 1000,
                                  save_steps=5000,
                                  save_total_limit=20,
                                  report_to='none',
                                )

In [21]:
trainer = Trainer(model=lora_model,
                  args = training_args,
                 train_dataset=tokenized_ds["train"],
                 eval_dataset=tokenized_ds["validation"],
                 compute_metrics=compute_metrics,
                 data_collator = data_collator)

No label_names provided for model class `PeftModelForSequenceClassification`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


In [22]:
result = trainer.train()

Step,Training Loss,Validation Loss,Accuracy
1000,0.735800,0.693763,0.695864
2000,0.550700,0.650769,0.711246
3000,0.534400,0.631658,0.725799
4000,0.522900,0.623928,0.724694
5000,0.520700,0.601735,0.738694
6000,0.510100,0.612114,0.738970
7000,0.517000,0.586860,0.746707
8000,0.504400,0.582677,0.748641
9000,0.501300,0.577811,0.753799
10000,0.501200,0.577328,0.749747


In [23]:
# Performance of the model before finetuning
training_args = TrainingArguments(
    per_device_eval_batch_size=32,
    report_to= "none",
    fp16=True
)

trainer = Trainer(
    model= lora_model,
    args = training_args,
    eval_dataset= tokenized_ds['test'],
    compute_metrics= compute_metrics,
    data_collator= data_collator
)

No label_names provided for model class `PeftModelForSequenceClassification`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


In [24]:
metrics = trainer.evaluate()
print(metrics)


{'eval_loss': 0.6313320994377136, 'eval_model_preparation_time': 0.0, 'eval_accuracy': 0.7254061527825786, 'eval_runtime': 30.0852, 'eval_samples_per_second': 480.801, 'eval_steps_per_second': 15.057}
